In [14]:
from datasets import load_dataset

# 加载 Hugging Face 上的指定数据集的 train 划分
dataset = load_dataset("/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA-Corpus", split="train")
# 随机打乱（可选），然后选取前 1000 条
subset = dataset.shuffle(seed=42).select(range(5000))

# 保存子集为 Parquet，路径可以是文件或目录
subset.to_parquet("OpenDocVQA-Corpus/subset5000.parquet")


Creating parquet from Arrow format: 100%|██████████| 50/50 [00:05<00:00,  9.89ba/s]


2146640336

In [15]:
from datasets import load_dataset

# 从本地 Parquet 文件加载为 DatasetDict 或 Dataset
reloaded = load_dataset("parquet", data_files="OpenDocVQA-Corpus/subset5000.parquet")


Generating train split: 5000 examples [00:06, 750.69 examples/s]


In [ ]:

from PIL import Image
import time

# reloaded['train'][0]['image'].show()
def add_label(example,idx):
    print(idx)

    example["my_new_label"] = 1
    # time.sleep(0.5)
    return example
subset = subset.map(add_label,with_indices=True,) 
print(subset[0])

In [22]:
from datasets import load_dataset
from tqdm import tqdm

curpus_data = load_dataset("parquet", data_files="/home/yuhaiyang/zyl/dataset/OpenDocVQA/test/subset5000.parquet")['train']
curpus_data=curpus_data.train_test_split(0.1)
doc_ids_train=set()
doc_ids_test=set()
for i in tqdm(range(len(curpus_data['train']))):
    doc_ids_train.add(curpus_data['train'][i]['doc_id'])
for i in tqdm(range(len(curpus_data['test']))):
    doc_ids_test.add(curpus_data['test'][i]['doc_id'])
print(len(doc_ids_train))
print(len(doc_ids_test))
# print(QA_data[0])

 99%|█████████▊| 4436/4500 [01:24<00:00, 68.83it/s]/home/yuhaiyang/anaconda3/envs/VDocRAG/lib/python3.10/site-packages/PIL/Image.py:3442: DecompressionBombWarning: Image size (164475666 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
100%|██████████| 500/500 [00:10<00:00, 47.62it/s]

4500
500


In [17]:
import copy
QA_data = load_dataset("/home/yuhaiyang/zyl/dataset/OpenDocVQA/OpenDocVQA", split="train")
QA_data_copy=copy.deepcopy(QA_data)
def filter_by_doc_id(example,doc_ids):

    return example['relevant_doc_ids'][0] in doc_ids
print('before filter',len(QA_data))
QA_data_test=QA_data_copy.filter(filter_by_doc_id,fn_kwargs={'doc_ids':doc_ids_test})
QA_data_train=QA_data_copy.filter(filter_by_doc_id,fn_kwargs={'doc_ids':doc_ids_train})

print('test after filter',len(QA_data_test))
print('train after filter',len(QA_data_train))


before filter 41020
test after filter 1416
train after filter 14657


In [9]:

num_dict={doc_id:0 for doc_id in doc_ids_test}
for i in range(len(QA_data_copy)):
    doc_id= QA_data_copy[i]['relevant_doc_ids'][0]
    if doc_id in doc_ids_test:
        num_dict[doc_id]+=1
print(max(num_dict.values()))
print(min(num_dict.values()))
count=0
all_C=0
for k,v in num_dict.items():
    all_C+=v
    if v==0:
        count+=1
print(count)
print(all_C)

In [25]:
from datasets import concatenate_datasets, load_dataset
from datasets import load_dataset, DatasetDict
# print(QA_data_train)
QA_data_sub5000=DatasetDict({
    'train':QA_data_train,
    'test':QA_data_test
})
print(QA_data_sub5000)
# QA_data_train.map(lambda ex: {"split",'train'})
# QA_data_test.map(lambda ex: {"split",'test'})

# QA_data_sub5000=concatenate_datasets([QA_data_train,QA_data_test])
# print(QA_data_sub5000)
QA_data_sub5000['train'].to_parquet('output/QA_data_sub5000_train.parquet')
QA_data_sub5000['test'].to_parquet('output/QA_data_sub5000_test.parquet')

Creating parquet from Arrow format: 100%|██████████| 17/17 [00:00<00:00, 20.49ba/s]


4589883

In [ ]:
dataset = load_dataset('/home/yuhaiyang/zyl/code/VDocRAG-main/output',split='train')
print(len(dataset))